# 02 — Error analysis
Run after Phase 5 (`src.compare`). Shows where fine-tuning helped/hurt and the 10 worst failures.

In [ ]:
import csv, json
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

RESULTS = Path("../results")            # Colab: Path("/content/drive/MyDrive/docvqa-qlora/results")
DATA_DIR = Path("../data/processed")    # Colab: Path("/content/data/processed")
comp = json.load(open(RESULTS / "comparison.json"))
print(json.dumps({k: comp[k] for k in ("n", "baseline", "finetuned", "better", "worse", "same")}, indent=1))
summary = list(csv.DictReader(open(RESULTS / "error_summary.csv")))
for s in summary:
    print(f"{s['question_type']:>12} / {s['answer_length']:<10} n={s['n']:>4}  base={s['baseline_anls']}  ft={s['finetuned_anls']}  +{s['better']} -{s['worse']}")

In [ ]:
rows = list(csv.DictReader(open(RESULTS / "error_analysis.csv")))
groups = [s for s in summary if s["question_type"] != "ALL" and s["answer_length"] == "ALL"]
x = range(len(groups))
plt.bar([i - .2 for i in x], [float(g["baseline_anls"]) for g in groups], .4, label="baseline")
plt.bar([i + .2 for i in x], [float(g["finetuned_anls"]) for g in groups], .4, label="fine-tuned")
plt.xticks(list(x), [g["question_type"] for g in groups]); plt.ylabel("ANLS"); plt.legend(); plt.title("ANLS by question type"); plt.show()

In [ ]:
test = {str(json.loads(l)["id"]): json.loads(l) for l in open(DATA_DIR / "test.jsonl")}
worst = sorted(rows, key=lambda r: (float(r["finetuned_anls"]), float(r["delta_anls"])))[:10]
for r in worst:
    t = test[r["id"]]
    fig = plt.figure(figsize=(7, 8)); plt.imshow(Image.open(DATA_DIR / t["image"]), cmap="gray"); plt.axis("off")
    plt.title(f"Q: {r['question']}\ngold: {r['gold']}\nfine-tuned: {r['finetuned_pred']} (ANLS {float(r['finetuned_anls']):.2f})"
              f" | baseline: {r['baseline_pred']}", fontsize=9, loc="left")
    plt.show()

In [ ]:
# Samples that got WORSE after fine-tuning
for r in [r for r in rows if r["status"] == "worse"][:15]:
    print(f"[{r['question_type']}] {r['question']}\n   gold={r['gold']!r}  base={r['baseline_pred']!r}  ft={r['finetuned_pred']!r}")